# El problema del transporte

En este notebook resolveremos un ejemplo del problema del transporte usando dos librerías diferente, usando **Scipy.optimize.linprog** y usando **PuLP**.


## Scipy.optimize.linprog:













## PuLP



## Ejemplo

MG Auto cuenta con tres plantas en Los Ángeles, Detroit y Nueva Orleáns, y dos importantes
centros de distribución en Denver y Miami. Las capacidades trimestrales de las tres plantas son
1000, 1500 y 1200 automóviles, y las demandas de los dos centros de distribución durante el
mismo periodo son de 2300 y 1400 automóviles. La distancia en millas entre las plantas y los centros
de distribución aparece en la la siguiente tabla

La compañía transportista cobra 8 centavos por milla por automóvil. En la tabla 5.2 se dan
los costos de transporte por automóvil en las diferentes rutas, redondeados al dólar más cercano.

Siguiendo el esquema general del modelo de transporte con m orígenes y n destinos como se muestra en la siguiente figura

La estructura especial del problema de transporte permite una representación compacta
del problema utilizando el formato tabla de transporte que aparece en la tabla 5.3.


Entonces el planteamineto del problema es

## Solución del problema

In [11]:
# importar bibliotecas
import numpy as np
from scipy.optimize import linprog

In [12]:
# Coeficientes de costos de transporte (en una matriz)
costos = np.array([[80,215],
                   [100,108],
                   [102,68]])
costos

array([[ 80, 215],
       [100, 108],
       [102,  68]])

In [13]:
# Capacidad de producción de las Plantas
capacidad_fabricas = [1000,1500,1200]

In [14]:
# Demanda de los centros de distribución
demanda_almacenes = [2300,1400]

In [15]:
# Número de fábricas y almacenes
num_fabricas, num_almacenes = costos.shape
costos.shape
#print(num_almacenes)

(3, 2)

In [16]:
# Reformula el problema para incluir restricciones de suministro y demanda
A_eq = np.zeros((num_fabricas + num_almacenes, num_fabricas * num_almacenes))
b_eq = np.zeros(num_fabricas + num_almacenes)
print(A_eq)
print(b_eq)

[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
[0. 0. 0. 0. 0.]


In [17]:
# Restricciones de suministro (capacidades de fábricas)
for i in range (num_fabricas):
  A_eq[i,i * num_almacenes:(i + 1) * num_almacenes] = 1
  b_eq[i] = capacidad_fabricas[i]

print(A_eq)
print(b_eq)

[[1. 1. 0. 0. 0. 0.]
 [0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
[1000. 1500. 1200.    0.    0.]


In [18]:

# Restricciones de demanda (demanda de almacenes)
for i in range (num_almacenes):
  A_eq[num_fabricas + i, i::num_almacenes] = 1
  b_eq[num_fabricas + i] = demanda_almacenes[i]

print(A_eq)
print(b_eq)

[[1. 1. 0. 0. 0. 0.]
 [0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 1. 1.]
 [1. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 1.]]
[1000. 1500. 1200. 2300. 1400.]


In [19]:
# Resuelve el problema de transporte utilizando linprog
resultado = linprog(costos.flatten(),  # Función objetivo (costos como coeficientes)
                    A_eq=A_eq,  # Restricciones de igualdad (suministro y demanda)
                    b_eq=b_eq,  # Lados derechos de las restricciones de igualdad
                    bounds=(0, None),  # Limitación de envío mínimo a 0, envío máximo no restringido
                    method='highs')  # Método de resolución


In [20]:
# Obtiene la matriz de asignación
asignacion = resultado.x.reshape((num_fabricas, num_almacenes))
print("Matriz de asignación")
print(asignacion)

Matriz de asignación
[[1000.    0.]
 [1300.  200.]
 [   0. 1200.]]


In [21]:
# Calcula el costo total de transporte
costo_total = np.sum(asignacion * costos)
print("Costo total de transporte:", costo_total)

Costo total de transporte: 313200.0


## Utilizando la biblioteca PuLP

In [22]:
#!pip install pulp
!pip install pulp
import pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 75.2 MB/s eta 0:00:00


In [23]:
# Definir las fuentes, destinos, ofertas y demandas
fuentes = ["LA", "Detroit","Norleans"]
destinos = ["Denver","Miami"]
ofertas = [1000,1500,1200]
demandas = [2300,1400]

In [24]:
# Definir los costos de transporte en una matriz
costos = [
    [80,215],
    [100,108],
    [102,68]
]

In [25]:
# Crear el problema de transporte
problema = pulp.LpProblem("Problema_de_Transporte", pulp.LpMinimize)


In [26]:
# Crear las variables de decisión (cantidades a transportar)
cantidades = pulp.LpVariable.dicts("Cantidad", (fuentes, destinos), lowBound=0, cat = "Integer")

In [27]:
# Definir la función objetivo
problema += pulp.lpSum(costos[i][j]* cantidades[f][d] for i, f in enumerate(fuentes) for j, d in enumerate(destinos))

In [28]:
# Restricciones de oferta
for i, f in enumerate(fuentes):
    problema += pulp.lpSum(cantidades[f][d] for d in destinos) == ofertas[i]

In [29]:
# Restricciones de demanda
for j, d in enumerate(destinos):
    problema += pulp.lpSum(cantidades[f][d] for f in fuentes) == demandas[j]

In [30]:
# Resolver el problema
problema.solve()

1

In [31]:
# Imprimir resultados
print("Estado:", pulp.LpStatus[problema.status])
print("Costo total de transporte:", pulp.value(problema.objective))
for f in fuentes:
    for d in destinos:
        print(f"Cantidad de {f} a {d}: {cantidades[f][d].varValue}")

Estado: Optimal
Costo total de transporte: 313200.0
Cantidad de LA a Denver: 1000.0
Cantidad de LA a Miami: 0.0
Cantidad de Detroit a Denver: 1300.0
Cantidad de Detroit a Miami: 200.0
Cantidad de Norleans a Denver: 0.0
Cantidad de Norleans a Miami: 1200.0


In [32]:
#!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic